# Assignment: Build an Autonomous AI Agent

This notebook builds an **AI agent** that decides by itself which **tool** to use.

- **Tools:** a safe calculator and a mock web search
- **Prompt template:** the rules that tell the agent when to use each tool
- **Memory:** lets follow-up questions refer to earlier answers
- **ReAct loop:** the agent *reasons* (Thought), *acts* (Action), reads the result (Observation) and repeats

In [1]:
# Step 0: Install the libraries (only needed once)
%pip install -q "langchain-classic>=1.0" "langchain-google-genai>=3.0"

Note: you may need to restart the kernel to use updated packages.


## Step 1: Imports and the LLM
The **LLM** is the agent's brain: it reads the question and decides what to do.

In [2]:
# Step 1: Imports and the LLM
# LangChain 1.x moved the classic agent API (initialize_agent, Tool, memory) to `langchain_classic`.
import ast
import math
import operator
import os
import re
import warnings
from getpass import getpass

from langchain_classic.agents import AgentType, ConversationalAgent, Tool, initialize_agent
from langchain_classic.agents.conversational.output_parser import ConvoOutputParser
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.agents import AgentAction
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

warnings.filterwarnings("ignore", category=DeprecationWarning)  # hide deprecation notices to keep the output clean

# LLM = the agent's "brain". Any chat model works; to use OpenAI, change only this line.
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google AI Studio API key: ")

# temperature=0 -> same input, same answer (repeatable tool choices)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

## Step 2: Tools
A **tool** is a normal Python function plus a **description**. The agent reads the description to choose the right tool.

In [3]:
# Step 2: The tools (Python functions the agent can call)

# --- 2a. Calculator ---
# eval() runs ANY Python code: dangerous with text written by an LLM.
# So we first parse the text into an AST (a tree of the expression)
# and only allow numbers, math operators and a few math functions.

ALLOWED_FUNCTIONS = {
    "sqrt": math.sqrt, "log": math.log, "log10": math.log10, "exp": math.exp,
    "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "abs": abs, "round": round, "min": min, "max": max,
}
ALLOWED_CONSTANTS = {"pi": math.pi, "e": math.e}
ALLOWED_NODES = (
    ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.Call, ast.Name, ast.Load,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow, ast.USub, ast.UAdd,
)


def calculator_tool(expression: str) -> str:
    """Evaluate a math expression safely and return the result as text."""
    # Clean up common LLM formats: "1,000", "2^10", "23 x 5"
    expr = expression.strip().strip("\"'`").rstrip("=").strip()
    expr = re.sub(r"(?<=\d),(?=\d{3})", "", expr)   # remove thousands separators
    expr = expr.replace("^", "**").replace("×", "*").replace("÷", "/")
    expr = re.sub(r"(?<=\d)\s*[xX]\s*(?=\d)", "*", expr)

    try:
        tree = ast.parse(expr, mode="eval")
        for node in ast.walk(tree):
            if not isinstance(node, ALLOWED_NODES):
                raise ValueError(f"'{type(node).__name__}' is not allowed")
            if isinstance(node, ast.Constant) and not isinstance(node.value, (int, float)):
                raise ValueError("only numbers are allowed")
            if isinstance(node, ast.Name) and node.id not in ALLOWED_FUNCTIONS | ALLOWED_CONSTANTS:
                raise ValueError(f"unknown name '{node.id}'")
            if isinstance(node, ast.Call) and not isinstance(node.func, ast.Name):
                raise ValueError("only simple function calls are allowed")
            if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Pow):
                if isinstance(node.right, ast.Constant) and abs(node.right.value) > 100:
                    raise ValueError("exponent too large")
        result = eval(compile(tree, "<calculator>", "eval"),
                      {"__builtins__": {}}, {**ALLOWED_FUNCTIONS, **ALLOWED_CONSTANTS})
    except ZeroDivisionError:
        return "Error: division by zero."
    except Exception as exc:  # the error goes back to the agent as an Observation, so it can retry
        return f"Error: could not evaluate '{expression}' ({exc}). Send a plain arithmetic expression like '23 * 5'."

    if isinstance(result, float) and result.is_integer():
        result = int(result)
    return f"{result:,}" if isinstance(result, int) else f"{result:,.6g}"


# --- 2b. Mock search ---
# A small in-memory "index" instead of a real search API.
# It matches by shared keywords, so different phrasings find the same result.
MOCK_SEARCH_INDEX = [
    ({"capital", "france"},
     "Paris is the capital and largest city of France, with about 2.1 million residents "
     "(11 million in the metro area). Source: mock-encyclopedia.org"),
    ({"population", "india", "people"},
     "India's population is estimated at 1,428,627,663 (UN World Population Prospects, 2023), "
     "making it the most populous country in the world. Source: mock-un-data.org"),
    ({"population", "france", "people"},
     "France has a population of about 68,170,000 (INSEE, January 2024). Source: mock-insee.fr"),
    ({"eiffel", "tower", "height", "tall"},
     "The Eiffel Tower is 330 metres (1,083 ft) tall including antennas. Source: mock-toureiffel.paris"),
    ({"capital", "japan", "tokyo"},
     "Tokyo is the capital of Japan, with about 14 million residents. Source: mock-encyclopedia.org"),
    ({"speed", "light"},
     "The speed of light in vacuum is exactly 299,792,458 metres per second. Source: mock-physics.org"),
]
STOPWORDS = {"what", "is", "the", "of", "a", "an", "in", "for", "how", "many", "who", "search", "find"}


def mock_search_tool(query: str) -> str:
    """Return a canned, realistic-looking search snippet for the query."""
    words = set(re.findall(r"[a-z]+", query.lower())) - STOPWORDS
    best_score, best_snippet = 0, None
    for keywords, snippet in MOCK_SEARCH_INDEX:
        score = len(words & keywords)
        if score > best_score:
            best_score, best_snippet = score, snippet
    if best_snippet is None:
        return f"[Mock search] No results found for '{query}'."
    return f"[Mock search] {best_snippet}"


# --- 2c. Wrap the functions as LangChain Tools ---
# The LLM never sees the code: it picks a tool by reading its description.
# So each description says WHEN to use it and WHAT input to send.
tools = [
    Tool(
        name="Calculator",
        func=calculator_tool,
        description=(
            "Use for ANY arithmetic or math: addition, subtraction, multiplication, division, "
            "powers, percentages, square roots. Input must be a single plain math expression "
            "using numbers only, e.g. '23 * 5' or '1428627663 + 34'. Never send words."
        ),
    ),
    Tool(
        name="Search",
        func=mock_search_tool,
        description=(
            "Use to look up facts about the world: capitals, populations, places, people, "
            "measurements, current information. Input should be a short search query, "
            "e.g. 'population of India'. Do not use it for math."
        ),
    ),
]

# Test the tools alone (no LLM yet)
print("Calculator('23 * 5')                  ->", calculator_tool("23 * 5"))
print("Calculator('1,428,627,663 + 34')      ->", calculator_tool("1,428,627,663 + 34"))
print("Calculator('sqrt(144) + 2^3')         ->", calculator_tool("sqrt(144) + 2^3"))
print("Calculator(\"__import__('os')\")      ->", calculator_tool("__import__('os').system('dir')"))
print("Search('capital of France')           ->", mock_search_tool("capital of France"))
print("Search('population India')            ->", mock_search_tool("population India"))

Calculator('23 * 5')                  -> 115
Calculator('1,428,627,663 + 34')      -> 1,428,627,697
Calculator('sqrt(144) + 2^3')         -> 20
Calculator("__import__('os')")      -> Error: could not evaluate '__import__('os').system('dir')' (only simple function calls are allowed). Send a plain arithmetic expression like '23 * 5'.
Search('capital of France')           -> [Mock search] Paris is the capital and largest city of France, with about 2.1 million residents (11 million in the metro area). Source: mock-encyclopedia.org
Search('population India')            -> [Mock search] India's population is estimated at 1,428,627,663 (UN World Population Prospects, 2023), making it the most populous country in the world. Source: mock-un-data.org


## Step 3: Prompt template
The **prompt template** is the agent's instruction sheet: its role, its rules and where the memory goes.

In [4]:
# Step 3: The prompt template (the agent's instructions)
# Final prompt = PREFIX + tool list + format rules + SUFFIX
#   PREFIX: the agent's role + rules for choosing a tool
#   SUFFIX: chat history (memory) + new question + scratchpad (steps taken so far)

PREFIX = """You are a careful research-and-math assistant. You answer questions by choosing the right tool.

Tool selection rules:
1. If the question needs ANY arithmetic, use Calculator. Never do math in your head, even simple math.
2. If the question needs a fact about the world (capitals, populations, places, measurements), use Search.
   Never answer factual questions from memory.
3. If the question needs BOTH, break it into steps: first Search for the missing fact, then pass the
   number you found to Calculator. Use one tool per step.
4. If the question refers to something said earlier ("that number", "the previous result", "my first
   question"), read the previous conversation history below and reuse that value instead of asking again.
5. Greetings or questions about the conversation itself need no tool.
6. Keep the final answer short, state the number or fact clearly, and mention which tool(s) you used.

TOOLS:
------
You have access to the following tools:"""

SUFFIX = """Begin!

Previous conversation history:
{chat_history}

New input: {input}
{agent_scratchpad}"""

# Build the template so we can print and inspect it
prompt = ConversationalAgent.create_prompt(
    tools,
    prefix=PREFIX,
    suffix=SUFFIX,
    input_variables=["input", "chat_history", "agent_scratchpad"],
)
assert isinstance(prompt, PromptTemplate)
print("Input variables:", prompt.input_variables)
print("=" * 90)
print(prompt.template)

Input variables: ['agent_scratchpad', 'chat_history', 'input']
You are a careful research-and-math assistant. You answer questions by choosing the right tool.

Tool selection rules:
1. If the question needs ANY arithmetic, use Calculator. Never do math in your head, even simple math.
2. If the question needs a fact about the world (capitals, populations, places, measurements), use Search.
   Never answer factual questions from memory.
3. If the question needs BOTH, break it into steps: first Search for the missing fact, then pass the
   number you found to Calculator. Use one tool per step.
4. If the question refers to something said earlier ("that number", "the previous result", "my first
   question"), read the previous conversation history below and reuse that value instead of asking again.
5. Greetings or questions about the conversation itself need no tool.
6. Keep the final answer short, state the number or fact clearly, and mention which tool(s) you used.

TOOLS:
------
You have

## Step 4: The agent
`initialize_agent` connects everything: **LLM + tools + memory + prompt**, running in a **ReAct** loop.

In [5]:
# Step 4: Create the agent = LLM + tools + memory + prompt

# Memory: saves each question/answer and adds it to {chat_history} on the next turn
memory = ConversationBufferMemory(memory_key="chat_history")


# Output parser = reads the LLM's text and decides: call a tool, or finish.
# Problem: Gemini sometimes writes an Action AND invents its result; the default parser accepted it.
# Fix: always run the FIRST Action and ignore the rest -> answers come from real tool results.
class FirstActionParser(ConvoOutputParser):
    def parse(self, text):
        match = re.search(r"Action:\s*(.*?)\s*Action Input:\s*(.*?)\s*(?=Thought:|Observation:|\n|$)",
                          text, re.DOTALL)
        if match:
            return AgentAction(match.group(1).strip(), match.group(2).strip(" \"'"), text[:match.end()])
        return super().parse(text)


agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,   # ReAct = Reason + Act loop, with chat history
    memory=memory,
    agent_kwargs={"prefix": PREFIX, "suffix": SUFFIX,     # our prompt from Step 3 + the custom parser
                  "output_parser": FirstActionParser()},
    verbose=True,                                         # show every Thought / Action / Observation
    handle_parsing_errors=True,                           # if the LLM breaks the format, retry instead of crashing
    max_iterations=6,                                     # stop after 6 steps (avoids infinite loops)
)

# Check: the agent really uses our template from Step 3
assert agent.agent.llm_chain.prompt.template == prompt.template
print("Agent ready with tools:", [t.name for t in tools])

Agent ready with tools: ['Calculator', 'Search']


## Step 5: Tests
Each test checks one behavior: a tool, a fact, two tools at once, and memory.

In [6]:
# Step 5: Test the agent
# ask() sends one question and prints the agent's final answer

def ask(question: str) -> str:
    print(f"\n{'#' * 90}\nUSER: {question}\n{'#' * 90}")
    answer = agent.invoke({"input": question})["output"]
    print(f"\nFINAL ANSWER: {answer}")
    return answer

In [7]:
# Test 1: math -> the agent should use the Calculator
ask("What is 23 * 5?")


##########################################################################################
USER: What is 23 * 5?
##########################################################################################


> Entering new AgentExecutor chain...


Thought: Do I need to use a tool? Yes
Action: Calculator
Action Input: 23 * 5
Observation: 115
Thought:

Thought: Do I need to use a tool? No
AI: 23 * 5 = 115. I used the Calculator tool to find this result.

> Finished chain.

FINAL ANSWER: 23 * 5 = 115. I used the Calculator tool to find this result.


'23 * 5 = 115. I used the Calculator tool to find this result.'

In [8]:
# Test 2: a fact -> the agent should use Search
ask("What is the capital of France?")


##########################################################################################
USER: What is the capital of France?
##########################################################################################


> Entering new AgentExecutor chain...


Thought: Do I need to use a tool? Yes
Action: Search
Action Input: capital of France
Observation: [Mock search] Paris is the capital and largest city of France, with about 2.1 million residents (11 million in the metro area). Source: mock-encyclopedia.org
Thought:

Do I need to use a tool? No
AI: The capital of France is Paris. I used the Search tool to find this fact.

> Finished chain.

FINAL ANSWER: The capital of France is Paris. I used the Search tool to find this fact.


'The capital of France is Paris. I used the Search tool to find this fact.'

In [9]:
# Test 3: two tasks in one question -> two tools in one run
ask("Add 34 and search for population of India.")


##########################################################################################
USER: Add 34 and search for population of India.
##########################################################################################


> Entering new AgentExecutor chain...


Thought: The user wants to add 34 to the previous mathematical result (115) and also search for the population of India. I need to break this down into steps, using one tool per step. First, I will use Calculator to compute 115 + 34.

Action: Calculator
Action Input: 115 + 34
Observation: 149
Thought:

Action: Search
Action Input: population of India
Observation: [Mock search] India's population is estimated at 1,428,627,663 (UN World Population Prospects, 2023), making it the most populous country in the world. Source: mock-un-data.org
Thought:

Thought: Do I need to use a tool? No
AI: Adding 34 to the previous result (115) gives 149. The estimated population of India is 1,428,627,663. I used the Calculator and Search tools to find these results.

> Finished chain.

FINAL ANSWER: Adding 34 to the previous result (115) gives 149. The estimated population of India is 1,428,627,663. I used the Calculator and Search tools to find these results.


'Adding 34 to the previous result (115) gives 149. The estimated population of India is 1,428,627,663. I used the Calculator and Search tools to find these results.'

In [10]:
# Test 4: memory -> "my first question" is read from the chat history
ask("Multiply the result of my first question by 2.")


##########################################################################################
USER: Multiply the result of my first question by 2.
##########################################################################################


> Entering new AgentExecutor chain...


Thought: Do I need to use a tool? Yes
Action: Calculator
Action Input: 115 * 2 
Observation: 230
Thought:

Do I need to use a tool? No
AI: Multiplying the result of your first question (115) by 2 gives 230. I used the Calculator tool to find this result.

> Finished chain.

FINAL ANSWER: Multiplying the result of your first question (115) by 2 gives 230. I used the Calculator tool to find this result.


'Multiplying the result of your first question (115) by 2 gives 230. I used the Calculator tool to find this result.'

In [11]:
# The memory: this text is added to {chat_history} on every call
print(memory.buffer)

Human: What is 23 * 5?
AI: 23 * 5 = 115. I used the Calculator tool to find this result.
Human: What is the capital of France?
AI: The capital of France is Paris. I used the Search tool to find this fact.
Human: Add 34 and search for population of India.
AI: Adding 34 to the previous result (115) gives 149. The estimated population of India is 1,428,627,663. I used the Calculator and Search tools to find these results.
Human: Multiply the result of my first question by 2.
AI: Multiplying the result of your first question (115) by 2 gives 230. I used the Calculator tool to find this result.


## Summary

| Test | Input | Tool(s) chosen |
|---|---|---|
| 1 | What is 23 * 5? | Calculator (115) |
| 2 | What is the capital of France? | Search (Paris) |
| 3 | Add 34 and search for population of India. | Memory -> Calculator (115 + 34 = 149) -> Search |
| 4 | Multiply the result of my first question by 2. | Memory -> Calculator (115 * 2 = 230) |

Good luck! Share screenshots or results as part of your submission.